In [8]:
import os
import voyageai
from pinecone import Pinecone
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# 1. Setup Clients
vo = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
index = pc.Index("japanese-wiki-index")

# 2. Your Test Question (Ask in English!)
query1 =  "What type of vehicle is Hermes from Kino's Journey"
query2 = "Who is the fastest horse in Uma Musume?"
query3 = "Tell me about Rin Tohsaka"
query4 = "Who are the main characters of Kino's Journey?"
query5 = "Who are the main characters of Bunny Drop or Usagi Drop?"
query6 = "Who is the tank in Kino's Journey?"
query = query6

# 3. Embed the query (IMPORTANT: use input_type="query")
query_emb = vo.embed([query], model="voyage-4-lite", input_type="query").embeddings[0]

# 4. Search Pinecone
results = index.query(vector=query_emb, top_k=3, include_metadata=True)

# 5. Show the results
print(f"--- Search Results for: '{query}' ---")
for match in results["matches"]:
    # Here is the 'Title Injection' we talked about!
    print(f"\n[Score: {match.score:.4f}]")
    print(f"SOURCE: {match.metadata['source']}")
    print(f"MEDIA TYPE: {match.metadata['media_type']}")
    print("-" * 30)
    print(match.metadata['text'][:400] + "...") # Show first 400 chars



# 6. Prepare the Context from Pinecone results
context_list = []
for match in results["matches"]:
    context_list.append(f"Source: {match.metadata['source']}\nContent: {match.metadata['text']}")

context_text = "\n\n---\n\n".join(context_list)

# 7. Create the Prompt
prompt = f"""
Answer in English. You are a helpful assistant. Answer the question based ONLY on the context provided below. If the answer isn't in the context, say you don't know.

Context:
{context_text}

Question: {query}
Answer:"""

# 8. Generate Answer (Example using OpenAI - requires 'openai' library)
response = client.chat.completions.create(
    model="gpt-4o-mini", # Fast and cheap for RAG
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

print("\n--- FINAL ANSWER ---")
print(response.choices[0].message.content)

--- Search Results for: 'Who is the tank in Kino's Journey?' ---

[Score: 0.5335]
SOURCE: ポップンタンクス!.txt
MEDIA TYPE: game
------------------------------
タンクワールド
「タンクワールド」では、プレイヤー自身がタンク乗りとなってタンク競技のチャンピオンを目指すモードである。内容としては競技に参加するNPCらと対戦を行ってタンクをカスタムする「パーツ」を集め、性能や戦績を高めてランキングの上位を目指してゆく。なお、当モードでそれぞれがカスタムしたタンクのデータは持ち寄って対戦することもできる。

キャラクター／搭乗戦車
一人用のストーリーモードにおいては、プチタンクを軍事目的で利用する「タリン帝国」と9人のタンク乗りとの騒動が描かれる。

プエル／プチMk-1
声：くまいもとこ
町はずれで父親と自動車整備工場を営む少年。11歳。
ストーリーモードでは、タリン帝国の試作兵器とは知らずに廃棄されていたジャンクパーツから「プチMk-1」を組み立てて愛車としたことで、騒動に巻き込まれる様子が描かれている。
プエラ／フォアグラン
声：大谷育江
プエルの...

[Score: 0.5265]
SOURCE: 学園キノ.txt
MEDIA TYPE: manga
------------------------------
声 - 緒方恵美（第2作）
「雲の前で」でフォトと出会った喋るモトラド（二輪車）。「フォトの日々」以降のフォトが主人公の話では彼が語り手となる。
元は商隊のトラックに積まれていた商品。持ち主である商隊が全滅した後、唯一生き残っていた奴隷であるフォトに話しかける。フォトの事情を承知しており、絶望し死にたがっていた彼女を諭し、二人で生き延びるべく尽力する。
一人称は「オレ」。高く透き通った男性の声で喋るもののガラが悪さとかなりの毒舌家である。しかし口の悪さに反して面倒見は良く、世渡りに向かないフォトのことを気にかけて指南役を続けている。折り畳み式の超小型のモトラドで、その特殊な形状の為か買い手がつかなかった経緯があり、本人もその事を気にしている節がある。同じモドラトのエルメスと比べると性悪的な思考をしており、きつい言